# Time Series Practice 3
## Phase 1 — Data Preparation

Notebook này trình bày và kiểm chứng kết quả Phase 1. Toàn bộ logic xử lý chính thức được gọi từ `src/practice_3/data/data_preparation.py`; notebook không duy trì một bản cài đặt thứ hai.

### 1. Bài toán, input và output

- **Bài toán:** chuẩn hóa chuỗi điện năng AEP theo từng giờ trước khi tạo đặc trưng.
- **Input:** `datasets/raw/AEP_hourly.csv`, gồm `Datetime` và `AEP_MW`.
- **Output dữ liệu:** `datasets/processed/aep_hourly_prepared.csv`.
- **Output kiểm toán:** `outputs/results/phase_01_data_quality.json`.
- **Phạm vi:** parse, sắp xếp, gộp timestamp trùng bằng trung bình, hoàn chỉnh timeline theo giờ và giữ nguyên target thiếu dưới dạng `NaN`.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

from practice_3.data.data_preparation import (
    DATETIME_FORMAT,
    EXPECTED_RAW_SHA256,
    calculate_sha256,
    run_phase_01,
)

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_path = project_root / "datasets/raw/AEP_hourly.csv"
processed_path = project_root / "datasets/processed/aep_hourly_prepared.csv"
audit_path = project_root / "outputs/results/phase_01_data_quality.json"

assert raw_path.exists(), f"Không tìm thấy raw dataset: {raw_path}"
print(f"Project root: {project_root}")
print(f"Raw dataset: {raw_path}")

Project root: /Users/vientu/Time Series/TOTAL_LAB_PRACTICE_FOR_TIME_SERIES/total_practice/Practice_3
Raw dataset: /Users/vientu/Time Series/TOTAL_LAB_PRACTICE_FOR_TIME_SERIES/total_practice/Practice_3/datasets/raw/AEP_hourly.csv


### 2. Kiểm tra dữ liệu raw

File raw chỉ được đọc để kiểm tra; pipeline không sửa file này.

In [2]:
df_raw = pd.read_csv(raw_path)
raw_summary = pd.DataFrame(
    {
        "Giá trị": [
            len(df_raw),
            len(df_raw.columns),
            int(df_raw.isna().sum().sum()),
            calculate_sha256(raw_path),
        ]
    },
    index=["Số dòng", "Số cột", "Tổng ô trống", "SHA-256"],
)
display(raw_summary)
display(df_raw.head())

,Giá trị
Số dòng,121273
Số cột,2
Tổng ô trống,0
SHA-256,109d122f7b485555c609eecdec2cd5a03172e0a08acd35...


,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


### 3. Chạy pipeline Phase 1

Pipeline áp dụng đúng contract đã duyệt: strict datetime parsing, stable sort, lấy trung bình tại timestamp trùng, reindex theo giờ, không nội suy target và giữ timestamp timezone-naive.

In [3]:
audit = run_phase_01(
    raw_path=raw_path,
    processed_path=processed_path,
    audit_path=audit_path,
)
print("Phase 1 pipeline completed.")
print(f"Processed dataset: {processed_path}")
print(f"Audit result: {audit_path}")

Phase 1 pipeline completed.
Processed dataset: /Users/vientu/Time Series/TOTAL_LAB_PRACTICE_FOR_TIME_SERIES/total_practice/Practice_3/datasets/processed/aep_hourly_prepared.csv
Audit result: /Users/vientu/Time Series/TOTAL_LAB_PRACTICE_FOR_TIME_SERIES/total_practice/Practice_3/outputs/results/phase_01_data_quality.json


### 4. Xem trước dữ liệu sau xử lý

In [4]:
df_prepared = pd.read_csv(processed_path)
df_prepared["Datetime"] = pd.to_datetime(
    df_prepared["Datetime"],
    format=DATETIME_FORMAT,
    errors="raise",
)
display(df_prepared.head())
display(df_prepared.tail())

,Datetime,AEP_MW,source_count,quality_flag
0,2004-10-01 01:00:00,12379.0,1,observed
1,2004-10-01 02:00:00,11935.0,1,observed
2,2004-10-01 03:00:00,11692.0,1,observed
3,2004-10-01 04:00:00,11597.0,1,observed
4,2004-10-01 05:00:00,11681.0,1,observed


,Datetime,AEP_MW,source_count,quality_flag
121291,2018-08-02 20:00:00,17673.0,1,observed
121292,2018-08-02 21:00:00,17303.0,1,observed
121293,2018-08-02 22:00:00,17001.0,1,observed
121294,2018-08-02 23:00:00,15964.0,1,observed
121295,2018-08-03 00:00:00,14809.0,1,observed


### 5. Kết quả kiểm toán dữ liệu

In [5]:
with audit_path.open(encoding="utf-8") as file:
    saved_audit = json.load(file)

audit_table = pd.DataFrame(
    {"Chỉ số": list(saved_audit.keys()), "Kết quả": list(saved_audit.values())}
)
display(audit_table)

,Chỉ số,Kết quả
0,phase,phase_01_data_preparation
1,raw_rows,121273
2,raw_missing_AEP_MW,0
3,raw_non_positive_AEP_MW,0
4,duplicate_rows_before,8
5,duplicate_timestamps_before,4
6,missing_hourly_timestamps_before,27
7,prepared_rows,121296
8,unique_timestamps,121296
9,prepared_time_start,2004-10-01 01:00:00


### 6. Validation gate

Các assertion dưới đây kiểm tra những tiêu chí bàn giao quan trọng nhất của Phase 1.

In [6]:
assert saved_audit == audit
assert saved_audit["raw_rows"] == 121_273
assert saved_audit["prepared_rows"] == 121_296
assert saved_audit["duplicate_rows_before"] == 8
assert saved_audit["duplicate_timestamps_before"] == 4
assert saved_audit["missing_hourly_timestamps_before"] == 27
assert saved_audit["duplicate_timestamps_after"] == 0
assert saved_audit["missing_AEP_MW"] == 27
assert saved_audit["sum_source_count"] == 121_273
assert saved_audit["timeline_continuous_hourly"] is True
assert saved_audit["timezone_policy"] == "timezone-naive"
assert saved_audit["raw_sha256_before"] == EXPECTED_RAW_SHA256
assert saved_audit["raw_sha256_after"] == EXPECTED_RAW_SHA256
assert saved_audit["status"] == "PASS"
assert calculate_sha256(raw_path) == EXPECTED_RAW_SHA256
print("Phase 1 validation gate: PASS")

Phase 1 validation gate: PASS


### 7. Kết luận và giới hạn

- Phase 1 tạo được timeline liên tục gồm **121.296 timestamp giờ**.
- **4 timestamp trùng** (8 dòng raw) đã được gộp bằng trung bình và vẫn truy vết bằng `source_count=2`, `quality_flag=duplicate_aggregated`.
- **27 timestamp thiếu** được thêm vào timeline nhưng `AEP_MW` vẫn là `NaN`; không có target giả được tạo.
- Raw dataset giữ nguyên checksum và timestamp vẫn timezone-naive.
- Giới hạn: nguồn không cung cấp timezone/UTC offset; việc lấy trung bình không khôi phục được chính xác hai giờ vật lý có thể liên quan DST.
- Processed dataset đủ điều kiện làm input cho Phase 2 sau khi Phase 1 được ký PASS.